# Bayesian Network Lab: Likelihood and Inference
**Faisal Alburti | CS3081 Artificial Intelligence | Effat University**

The model follows the lab probability tables. All code cells were run in order, and their actual outputs are saved below. Run in Python 3.10+ with pgmpy installed. In a new environment, run `%pip install pgmpy==1.1.2` before running the model.

In [1]:
import platform, importlib.metadata
print("Python:", platform.python_version())
print("pgmpy:", importlib.metadata.version("pgmpy"))

Python: 3.13.18
pgmpy: 1.1.2


## 1-2. Build and validate the network
Rain affects Maintenance and Train; Maintenance affects Train; Train affects Appointment.

In [2]:
import warnings
warnings.filterwarnings("ignore")
from pgmpy.models import DiscreteBayesianNetwork
from pgmpy.factors.discrete import TabularCPD

model = DiscreteBayesianNetwork([
    ("rain", "maintenance"), ("rain", "train"),
    ("maintenance", "train"), ("train", "appointment")
])
rain = TabularCPD("rain", 3, [[0.7], [0.2], [0.1]],
    state_names={"rain": ["none", "light", "heavy"]})
maintenance = TabularCPD("maintenance", 2, [[0.4, 0.2, 0.1], [0.6, 0.8, 0.9]],
    evidence=["rain"], evidence_card=[3],
    state_names={"maintenance": ["yes", "no"], "rain": ["none", "light", "heavy"]})
# Columns: (none,yes), (none,no), (light,yes), (light,no), (heavy,yes), (heavy,no)
train = TabularCPD("train", 2,
    [[0.8, 0.9, 0.6, 0.7, 0.4, 0.5], [0.2, 0.1, 0.4, 0.3, 0.6, 0.5]],
    evidence=["rain", "maintenance"], evidence_card=[3, 2],
    state_names={"train": ["on time", "delayed"],
                 "rain": ["none", "light", "heavy"], "maintenance": ["yes", "no"]})
appointment = TabularCPD("appointment", 2, [[0.9, 0.6], [0.1, 0.4]],
    evidence=["train"], evidence_card=[2],
    state_names={"appointment": ["attend", "miss"], "train": ["on time", "delayed"]})
model.add_cpds(rain, maintenance, train, appointment)
print("Model is correct:", model.check_model())


Model is correct: True


## 3. Joint probability: normal day
$P(r,m,t,a)=P(r)P(m|r)P(t|r,m)P(a|t)$

$0.7 \times 0.6 \times 0.9 \times 0.9 = 0.3402$ (34.02%).

In [3]:
p = (rain.get_value(rain="none")
     * maintenance.get_value(maintenance="no", rain="none")
     * train.get_value(train="on time", rain="none", maintenance="no")
     * appointment.get_value(appointment="attend", train="on time"))
print(round(p, 4))

0.3402


## 4. Joint probability: bad day
$0.1 \times 0.1 \times 0.6 \times 0.4 = 0.0024$ (0.24%).

In [4]:
bad_day = (rain.get_value(rain="heavy")
     * maintenance.get_value(maintenance="yes", rain="heavy")
     * train.get_value(train="delayed", rain="heavy", maintenance="yes")
     * appointment.get_value(appointment="miss", train="delayed"))
print(round(bad_day, 4))

0.0024


## 5. Evidence: delayed train
A delay makes no rain less likely than its prior probability of 0.7.

In [5]:
from pgmpy.inference import VariableElimination
inference = VariableElimination(model)
def show(evidence):
    """Print each unobserved node distribution given the evidence."""
    for node in ["rain", "maintenance", "train", "appointment"]:
        if node in evidence:
            print(f"{node}: {evidence[node]}   (observed)")
            continue
        result = inference.query([node], evidence=evidence, show_progress=False)
        print(node)
        for value, prob in zip(result.state_names[node], result.values):
            print(f"    {value}: {prob:.4f}")
show({"train": "delayed"})

rain
    none: 0.4601
    light: 0.3005
    heavy: 0.2394
maintenance
    yes: 0.3662
    no: 0.6338
train: delayed   (observed)
appointment
    attend: 0.6000
    miss: 0.4000


## 6. Inference by enumeration
Sum over Maintenance. Appointment can be summed out because its two probabilities add to 1.

- None: $0.7(0.4(0.2)+0.6(0.1))=0.098$
- Light: $0.2(0.2(0.4)+0.8(0.3))=0.064$
- Heavy: $0.1(0.1(0.6)+0.9(0.5))=0.051$

$P(delayed)=0.098+0.064+0.051=0.213$. Normalize by dividing each value by 0.213.

In [6]:
weights = [0.7*(0.4*0.2 + 0.6*0.1),
           0.2*(0.2*0.4 + 0.8*0.3),
           0.1*(0.1*0.6 + 0.9*0.5)]
total = sum(weights)
print("Unnormalized:", [round(w, 3) for w in weights])
print(f"P(delayed): {total:.3f}")
print(f"alpha: {1/total:.4f}")
for state, weight in zip(["none", "light", "heavy"], weights):
    print(f"P({state} | delayed): {weight/total:.4f}")

Unnormalized: [0.098, 0.064, 0.051]
P(delayed): 0.213
alpha: 4.6948
P(none | delayed): 0.4601
P(light | delayed): 0.3005
P(heavy | delayed): 0.2394


## 7. Appointment
When Train is known to be delayed, the Appointment table gives attend = 0.6 and miss = 0.4.

## 8. Reasoning upward

In [7]:
show({"appointment": "miss"})

rain
    none: 0.6065
    light: 0.2392
    heavy: 0.1544
maintenance
    yes: 0.3441
    no: 0.6559
train
    on time: 0.4802
    delayed: 0.5198
appointment: miss   (observed)


## 9. Two observations
$P(yes|delayed,heavy)=\frac{0.1(0.6)}{0.1(0.6)+0.9(0.5)}=\frac{0.06}{0.51}\approx0.1176$.

In [8]:
show({"train": "delayed", "rain": "heavy"})
print(f"Hand check: {(0.1*0.6)/(0.1*0.6+0.9*0.5):.4f}")

rain: heavy   (observed)
maintenance
    yes: 0.1176
    no: 0.8824
train: delayed   (observed)
appointment
    attend: 0.6000
    miss: 0.4000
Hand check: 0.1176


## Short answers
**Q13. Why do the hand calculation and code agree?**  
Both use the same probability tables, sum over the hidden values, and normalize. Variable elimination reuses calculations but still gives the exact answer.

**Q14. Why is the probability of missing exactly 0.4?**  
Appointment has only Train as a parent. Once the train is known to be delayed, the miss probability is 0.4 directly from its table.

**Q16. Why does missing increase the probability of heavy rain?**  
Missing is evidence that a delay was more likely, and delays are more likely in heavy rain. This changes our belief about the rain; missing does not cause rain.

**Q18. Why does maintenance only increase from 0.1 to 0.1176?**  
With heavy rain, the delay probability is 0.6 with maintenance and 0.5 without it. Those numbers are close, so a delay gives only weak evidence for maintenance.


## Answer record
Setup: Shift + Enter. Q1: Rain and Maintenance. Q2: No, only through Train. Q3: 0.2. Q4: 0.5. Q5: 12. Q6: 0.7, 0.6, 0.9, 0.9. Q7: 0.3402. Q8: 0.0024. Q9: Goes down. Q10: 0.098, 0.064, 0.051. Q11: 0.213. Q12: 0.4601. Q13-Q14: See explanations above. Q15: 0.1544. Q16: See above. Q17: 0.1176. Q18: See above. Exit: Evidence can change beliefs above and below a node.

In [9]:
# Cross-check the numerical results against independent enumeration.
import math
assert model.check_model()
assert math.isclose(p, 0.3402)
assert math.isclose(bad_day, 0.0024)
assert math.isclose(total, 0.213)
posterior = inference.query(["rain"], evidence={"train":"delayed"}, show_progress=False)
assert all(math.isclose(float(a), b/total) for a,b in zip(posterior.values, weights))
assert math.isclose(sum(posterior.values), 1.0)
assert round(float(inference.query(["rain"], evidence={"appointment":"miss"}, show_progress=False).get_value(rain="heavy")), 4) == 0.1544
assert round(float(inference.query(["maintenance"], evidence={"rain":"heavy", "train":"delayed"}, show_progress=False).get_value(maintenance="yes")), 4) == 0.1176
print("All model and numerical checks passed.")

All model and numerical checks passed.
